<a href="https://colab.research.google.com/github/sankeawthong/Project-1-Lita-Chatbot/blob/main/Week01_Lab01_Colab%20and%20First%20Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 1 — Welcome to Colab & Your First Dataset

**Machine Learning (3 credits, 2-2-5) · Week 1**

| | |
|---|---|
| **Time budget** | ~2 hours (in-lab) |
| **Learning objectives** | 1. Operate a Colab/Jupyter notebook confidently (cells, kernel, saving) —  2. Apply the course working habits: save to Drive, load data by URL —  3. Perform a first exploratory data analysis: inspect, summarize, and plot a real dataset|
| **Dataset** | Palmer Penguins (Horst, Hill & Gorman, 2020) — 344 penguins from Palmer Station, Antarctica |

> **How labs work in this course:** the first half is a **guided walkthrough** — read, predict, run, compare. The second half is **exercise/self practice** you complete yourself. Green ✅ *self-check* cells tell you immediately whether you're on track.

---
## Part 0 · Colab

You are looking at a **notebook**: a document made of **cells**. Two kinds:

- **Markdown cells** (like this one) hold text.
- **Code cells** hold Python. Click one and press **`Shift+Enter`** to run it.

**!!Do this FIRST — before anything else:** go to **File → Save a copy in Drive**. Colab opened a *shared* copy of this notebook; your edits are not saved until you make your own copy. This is working habit #1, and forgetting it is the #1 way students lose work.

Filename at the top-left should now say "Copy of Lab1..." (rename it to `Lab1_<your_student_id>`). Now run your first cell:

In [ ]:
# Your first code cell. Click here, then press Shift+Enter.
import sys
print("Hello, Machine Learning!")
print("Python version:", sys.version.split()[0])

**Predict before you run** — that's the habit this course drills. Before running the next cell, answer in your head: what will it print?

In [ ]:
x = 10
x = x + 5
print(x)

### The kernel

Behind the notebook runs a Python process (the **kernel** / "runtime"). Variables live in the kernel, not in the cells — which means **execution order matters, not the order cells appear on the page**.

Run the next two cells **in order**, then run the *first* one again. Watch the value change. Notice the number in brackets `[n]` next to each cell — that's the execution counter, your audit trail.

In [ ]:
try:
    print("y is:", y)      # y doesn't exist yet...
except NameError as e:
    print("NameError:", e)
    print("^ Expected the FIRST time — y hasn't been defined. Run the NEXT cell, then re-run THIS one.")

In [ ]:
y = "defined now"
# Now scroll up and re-run the previous cell — no more error.

> **Rule:** if a notebook behaves strangely, the cure is **Runtime → Restart session and run all**. It replays the notebook top-to-bottom in a fresh kernel. Before you submit *any* lab, do exactly this — a notebook that only works because of leftover kernel state is a broken notebook.

**Tips:**
- `Ctrl+M B` inserts a cell below · `Ctrl+M D D` deletes a cell · `Ctrl+M M` converts to Markdown
- Colab sessions **disconnect when idle** and their temporary disk is wiped — your notebook survives (it's in Drive), but files you created on the session disk do not

### Working habit #2 — mounting Google Drive

Anything you want to **keep across sessions** (datasets you built, saved figures) goes in your Drive. The cell below connects it (Colab will ask for permission — approve it). On a lab machine instead of Colab, it politely skips itself.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Drive mounted at /content/drive — files saved there persist.")
except ImportError:
    print("Not running in Colab (lab machine / local Jupyter) — no Drive to mount. Carry on.")

---
## Setup — the standard course header

Every lab starts with a cell like this: imports, version printout, and a fixed **random seed**. (This week everything we need is preinstalled — later weeks add an install line here.)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RNG_SEED = 42
np.random.seed(RNG_SEED)

for name, mod in [("numpy", np), ("pandas", pd), ("seaborn", sns)]:
    print(f"{name:10s} {mod.__version__}")
print("Setup complete ✔")

---
## Part 1 · Meet the data — working habit #3: load by URL

Our first dataset: **344 penguins** measured at Palmer Station, Antarctica — bill and flipper dimensions, body mass, species, island, and sex. It's a genuine research dataset and the friendliest possible introduction to real data (including real *missing values*).

We load it **directly from a URL** — no downloads, no file-hunting, works identically on every machine. (Offline in the lab? Swap the URL for the local path in the comment.)

In [ ]:
PENGUINS_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"
# Offline fallback on lab machines: PENGUINS_URL = r"C:\ml-course\data\penguins.csv"

df = pd.read_csv(PENGUINS_URL)
df.head()          # the first 5 rows — always your first look at any dataset

Each **row** is one penguin (an *example*). Each **column** is a *feature* — remember the lecture table? There's no teal "label" column here because today we're not predicting anything; we're *exploring*. (Though keep an eye on `species`... it will become a label in a future week.)

**Predict:** how many rows and columns does `df` have? Commit to a guess, then run:

In [ ]:
print("shape (rows, columns):", df.shape)
df.info()

Read the `info()` output:
- **Dtype** — `float64` columns are numeric measurements; `object` columns are text categories. This distinction decides what math and which plots make sense.
- **Non-Null Count** — anything below 344 means **missing values**. Real data is holey; pretending otherwise is how models fail. Let's count the holes precisely:

In [ ]:
df.isna().sum()

A handful of missing measurements, and `sex` is missing for 11 birds. In Week 2 you'll learn the *repertoire* of responses (drop, fill, flag). Today we just refuse to be surprised by them.

Two more one-liners every data scientist runs on autopilot — numeric summaries, and category counts:

In [ ]:
df.describe().round(1)

In [ ]:
print(df["species"].value_counts(), "\n")
print(df["island"].value_counts())

---
## Part 2 · First plots — where the story lives

Tables tell you numbers; plots tell you *shape*. First: how is body mass distributed?

In [ ]:
plt.figure(figsize=(7, 4))
sns.histplot(data=df, x="body_mass_g", bins=25)
plt.title("Body mass of 344 Palmer penguins")
plt.xlabel("Body mass (g)")
plt.show()

**Look in the graph:** is that one hill or two? A lumpy histogram whispers that the data contains *groups*. Let's ask a second variable to separate them — flipper length vs body mass, colored by species:

In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(data=df, x="flipper_length_mm", y="body_mass_g",
                hue="species", alpha=0.8)
plt.title("Flipper length vs body mass, by species")
plt.show()

There it is — the species form **visibly distinct clouds**. Hold onto this picture: it is exactly why a classifier will one day predict species from measurements (supervised learning, Weeks 3+), and why a clustering algorithm could *discover* these groups without ever seeing the species column (unsupervised, Week 9). You have just seen, in one scatter plot, the geometric intuition behind half this course.

One more tool — `groupby`, the workhorse of "compare groups" questions:

In [ ]:
df.groupby("species")["body_mass_g"].mean().round(0)

---
---
# Self practice, Exercises

Rules of engagement: replace each `...` with your code · run the ✅ self-check after each exercise — it must print **passed** · AI assistance is allowed **with disclosure**.


### Exercise 1 — Counting
Create a variable **`island_counts`** holding the number of penguins on each island (a Series: island → count).

In [ ]:
island_counts = ...   # hint: you saw the tool for this in Part 1
island_counts

In [ ]:
# ✅ Self-check — Exercise 1
assert not isinstance(island_counts, type(Ellipsis)), "Replace the ... with your code first!"
assert int(island_counts.sum()) == 344, "Counts should total 344 penguins."
assert island_counts.idxmax() == "Biscoe", "Hmm — which island has the most penguins?"
print("Exercise 1 self-check passed ✔")

### Exercise 2 — Handling missing values
Create **`df_clean`**: a copy of `df` with every row containing *any* missing value removed. Then store the number of rows you lost in **`rows_dropped`** (an integer).

In [ ]:
df_clean = ...        # one pandas method does this
rows_dropped = ...    # compute it — don't hard-code the number
print(df_clean.shape, "| dropped:", rows_dropped)

In [ ]:
# ✅ Self-check — Exercise 2
assert df_clean.shape[0] == 333, "Expected 333 complete rows."
assert df_clean.isna().sum().sum() == 0, "df_clean still contains missing values."
assert rows_dropped == 11, "rows_dropped should be computed from the two DataFrames."
print("Exercise 2 self-check passed ✔")

### Exercise 3 — Group comparisons
Using **`df_clean`**, compute the **mean body mass for every (species, sex) combination** and store it in **`mass_by_group`**. Then put the name of the species containing the single heaviest group in **`heaviest_species`** (a string) — *computed from `mass_by_group`, not typed by eye.*

*Hint:* `groupby` accepts a list of columns; a Series has `.idxmax()`.

In [ ]:
mass_by_group = ...
heaviest_species = ...
print(mass_by_group.round(0), "\n\nHeaviest:", heaviest_species)

In [ ]:
# ✅ Self-check — Exercise 3
assert len(mass_by_group) == 6, "Expect 6 groups: 3 species × 2 sexes."
assert heaviest_species == "Gentoo", "Recompute — which species tops the table?"
assert abs(mass_by_group.max() - 5484.8) < 1.0, "The heaviest group's mean looks off."
print("Exercise 3 self-check passed ✔")

### Exercise 4 — A publication-worthy plot
Make a **scatter plot of `bill_length_mm` vs `bill_depth_mm`, colored by species**, using `df_clean`. Requirements: a descriptive **title**, readable **axis labels** (with units), and `figsize=(7, 5)`.

Then look at your plot and answer in **`bill_answer`**: within this plot, do the species separate into distinct clouds — `"yes"` or `"no"`?

In [ ]:
plt.figure(figsize=(7, 5))
...   # your scatter plot here (a very similar one appears in Part 2)
plt.show()

bill_answer = "..."   # "yes" or "no"

In [ ]:
# ✅ Self-check — Exercise 4 (checks the answer; the plot itself is graded by eye)
assert bill_answer in ("yes", "no"), 'Set bill_answer to "yes" or "no".'
assert bill_answer == "yes", "Look again — three species, three clouds?"
print("Exercise 4 self-check passed ✔  (make sure the plot has a title + labeled axes!)")

### Exercise 5 — Your first finding (open-ended)
Explore freely and find **one pattern in this dataset that we did not already show above**. Produce:
1. **One plot or table** that shows the pattern (code cell below), and
2. **Three or more sentences** in the Markdown cell after it: what you found, how the evidence shows it, and one honest caveat (what your evidence *cannot* tell you).

Ideas if you're stuck: differences between islands · male vs female measurement gaps · does bill shape differ where species overlap? — but your own question beats all of these.

In [ ]:
# Your exploration here
...

*Your finding (≥ 3 sentences): what you found · how the evidence shows it · one honest caveat.*

✍️ ...

---
## AI-usage disclosure

Edit the cell below honestly. Both extremes are perfectly acceptable answers; a missing or dishonest disclosure is not.

> **AI tools used:** *e.g., "Claude — explained what `.idxmax()` returns and helped debug a KeyError in Exercise 3" — or — "None."*
>
> **Written and understood by:** *your name & student ID*
>
> I can explain every line of code in this notebook.

## Submitting

1. **Runtime → Restart session and run all.** Every cell must run top-to-bottom without errors and every self-check must print *passed*.
2. Check the disclosure cell is filled in and the notebook is renamed `Lab1_<student_id>`.
3. **File → Download → Download .ipynb**, then submit before the deadline.

*Reminder: each week a few students are randomly selected to walk through their submission orally in the lab. Selection is genuinely random — being able to explain your own notebook is the only preparation.*

---
*Dataset: Horst A.M., Hill A.P., Gorman K.B. (2020). palmerpenguins. CC-0. Collected by Dr. Kristen Gorman, Palmer Station LTER.*